# Demonstration/Testing of Loading Data from S3 Buckets

This notebook demonstrates loading data from the lake: **bronze** (raw Statcast dailies), **processed** (legacy by-player paths), **silver** (engineered `player_year_features.parquet`), and **gold** (model-ready `player_year_features_preprocessed.parquet` after silver→gold preprocessing).

## Imports

In [ ]:
import pandas as pd
import s3fs
from datetime import date, timedelta

## Loading Data from Raw Statcast Data Bucket

In [ ]:
# Building our key
BUCKET = "diamond-dna"
PREFIX = "bronze/statcast"

# Start and End Date Selection
start_date = date(2025, 11, 1)
end_date   = date(2025, 11, 1)

# Function to build paths to date-by-date data
def build_daily_paths(bucket, prefix, start_date, end_date):
    paths = []
    current = start_date
    while current <= end_date:
        ds = current.strftime("%Y-%m-%d")
        year = current.year
        key = f"{prefix}/year={year}/date={ds}/statcast_pitches.parquet"
        paths.append(f"s3://{bucket}/{key}")
        current += timedelta(days=1)
    return paths

paths = build_daily_paths(BUCKET, PREFIX, start_date, end_date)

# Use AWS creds to access S3 buckets
fs = s3fs.S3FileSystem()

# Tell pandas/pyarrow to use the S3 filesystem
df = pd.read_parquet(paths, filesystem=fs)

pd.set_option('display.max_columns', None)
df.head()

FileNotFoundError: s3://diamond-dna/bronze/statcast/year=2025/date=2025-11-01/statcast_pitches.parquet

## Loading Data from Processed Statcast Data Bucket

### Pitcher

In [13]:
# Building our key (legacy by-player pitch parquet path; current pipeline uses bronze dailies + silver feature tables)
BUCKET = "diamond-dna"
PREFIX = "processed/statcast"
# Role selection
role = 'pitcher'

# Year selection
year = '2024'
# ID selection
id = 502171

# Use AWS creds to access S3 buckets
fs = s3fs.S3FileSystem()

# Read as a single file stream so PyArrow doesn't discover Hive partitioning.
# (Partition discovery would add a dictionary-encoded "year" that conflicts with the int32 "year" column inside the file.)
path = f"s3://{BUCKET}/{PREFIX}/pitcher/{role}_id={id}/year={year}/statcast_pitches.parquet"
with fs.open(path, "rb") as f:
    df = pd.read_parquet(f)

pd.set_option('display.max_columns', None)
df.head()

TypeError: ClientArgsCreator.compute_endpoint_resolver_builtin_defaults() missing 1 required positional argument: 's3_disable_express_session_auth'

### Batter

In [10]:
role = 'batter'
id = 444482

path = f"s3://{BUCKET}/{PREFIX}/batter/{role}_id={id}/year={year}/statcast_pitches.parquet"
with fs.open(path, "rb") as f:
    df = pd.read_parquet(f)

pd.set_option('display.max_columns', None)
df.head()


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,batter_name,pitcher_name,fielder_2_name,fielder_3_name,fielder_4_name,fielder_5_name,fielder_6_name,fielder_7_name,fielder_8_name,fielder_9_name,year
0,None,2024-03-16,<NA>,<NA>,<NA>,"Strickland, Hunter",444482,519326,strikeout,swinging_strike,<NA>,<NA>,<NA>,<NA>,<NA>,David Peralta strikes out swinging.,S,L,R,LAA,CHC,S,2,None,0,2,2024,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,8,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,747940,595453,676572,605346,691186,670869,666176,545350,666160,<NA>,<NA>,<NA>,0.0,<NA>,0,0,<NA>,62,3,None,2,4,4,2,4,2,4,2,None,None,<NA>,0.007,-0.162,<NA>,<NA>,<NA>,0.162,<NA>,-2,2,0.153,0.847,35,36,36,37,1,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"peralta, david","strickland, hunter","wallach, chad","wagaman, eric","lópez, jack",None,"soto, livan","adell, jo","marisnick, jake","moniak, mickey",2024
1,None,2024-03-16,<NA>,<NA>,<NA>,"Strickland, Hunter",444482,519326,None,swinging_strike,<NA>,<NA>,<NA>,<NA>,<NA>,Swinging Strike,S,L,R,LAA,CHC,S,<NA>,None,0,1,2024,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,8,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,747940,595453,676572,605346,691186,670869,666176,545350,666160,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,62,2,None,2,4,4,2,4,2,4,2,None,None,<NA>,0.002,-0.057,<NA>,<NA>,<NA>,0.057,<NA>,-2,2,0.151,0.849,35,36,36,37,1,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"peralta, david","strickland, hunter","wallach, chad","wagaman, eric","lópez, jack",None,"soto, livan","adell, jo","marisnick, jake","moniak, mickey",2024
2,None,2024-03-16,<NA>,<NA>,<NA>,"Strickland, Hunter",444482,519326,None,swinging_strike,<NA>,<NA>,<NA>,<NA>,<NA>,Swinging Strike,S,L,R,LAA,CHC,S,<NA>,None,0,0,2024,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,8,Top,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,747940,595453,676572,605346,691186,670869,666176,545350,666160,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,62,1,None,2,4,4,2,4,2,4,2,None,None,<NA>,0.001,-0.037,<NA>,<NA>,<NA>,0.037,<NA>,-2,2,0.15,0.85,35,36,36,37,1,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"peralta, david","strickland, hunter","wallach, chad","wagaman, eric","lópez, jack",None,"soto, livan","adell, jo","marisnick, jake","moniak, mickey",2024
3,None,2024-03-16,<NA>,<NA>,<NA>,

## Loading silver player-year feature tables


### Pitchers

In [26]:
BUCKET = "diamond-dna"
FEATURE_PREFIX = "silver"

role = "pitcher"
year = "2025"

fs = s3fs.S3FileSystem()
path = f"s3://{BUCKET}/{FEATURE_PREFIX}/{role}/year={year}/player_year_features.parquet"
with fs.open(path, "rb") as f:
    df = pd.read_parquet(f)

pd.set_option("display.max_columns", None)
df.head()

,role,player_id,year,n_pitches_total,batter_swing_rate,batter_zone_swing_rate,batter_chase_rate,batter_contact_rate,batter_whiff_rate,in_zone_rate,release_speed_max,fastball_velo_mean,offspeed_velo_mean,velo_differential,release_speed_iqr,release_spin_rate_iqr,pfx_x_iqr,release_extension_mean,release_extension_iqr,pfx_z_mean,pfx_z_iqr,plate_x_mean,plate_x_sd,plate_z_mean,plate_z_sd,edge_percent,meatball_percent,first_pitch_strike_rate,xwoba_allowed_lhb_mean,xwoba_allowed_rhb_mean,platoon_xwoba_allowed_diff,gb_percent_allowed,ld_percent_allowed,fb_percent_allowed,iffb_percent_allowed,delta_run_exp_mean,pt_CH_release_speed_mean,pt_CH_release_spin_rate_mean,pt_CH_pfx_x_mean,pt_FC_release_speed_mean,pt_FC_release_spin_rate_mean,pt_FC_pfx_x_mean,pt_CU_release_speed_mean,pt_CU_release_spin_rate_mean,pt_CU_pfx_x_mean,pt_SI_release_speed_mean,pt_SI_release_spin_rate_mean,pt_SI_pfx_x_mean,pt_FF_release_speed_mean,pt_FF_release_spin_rate_mean,pt_FF_pfx_x_mean,pt_NONE_release_speed_mean,pt_NONE_release_spin_rate_mean,pt_NONE_pfx_x_mean,pt_CS_release_speed_mean,pt_CS_release_spin_rate_mean,pt_CS_pfx_x_mean,pitch_type_CU_share,pitch_type_SI_share,pitch_type_FC_share,pitch_type_FF_share,pitch_type_CH_share,pitch_type_CS_share,pitch_type_SL_share,pitch_type_entropy,pt_SL_release_speed_mean,pt_SL_release_spin_rate_mean,pt_SL_pfx_x_mean,pt_ST_release_speed_mean,pt_ST_release_spin_rate_mean,pt_ST_pfx_x_mean,pitch_type_ST_share,pitch_type_PO_share,pt_KC_release_speed_mean,pt_KC_release_spin_rate_mean,pt_KC_pfx_x_mean,pt_FS_release_speed_mean,pt_FS_release_spin_rate_mean,pt_FS_pfx_x_mean,pitch_type_FS_share,pitch_type_KC_share,pitch_type_SV_share,pt_SV_release_speed_mean,pt_SV_release_spin_rate_mean,pt_SV_pfx_x_mean,pitch_type_FA_share,pitch_type_EP_share,pitch_type_UN_share,pitch_type_FO_share,pt_SC_release_speed_mean,pt_SC_release_spin_rate_mean,pt_SC_pfx_x_mean,pitch_type_SC_share,pt_KN_release_speed_mean,pt_KN_release_spin_rate_mean,pt_KN_pfx_x_mean,pitch_type_KN_share,pt_FO_release_speed_mean,pt_FO_release_spin_rate_mean,pt_FO_pfx_x_mean
0,pitcher,434378,2025,2775,0.483243,0.336577,0.146667,0.376937,0.106306,0.491892,98.3,93.926526,83.441730,10.484796,9.9,190.75,1.2700,6.016940,0.3,0.721243,1.300,0.113814,0.774437,2.469420,0.992253,0.833700,0.166300,0.623762,0.373694,0.361138,0.012556,0.357430,0.251004,0.303213,0.088353,-0.000224,84.695614,1723.048246,-1.079561,NaN,NaN,NaN,78.457403,2746.644156,0.625481,NaN,NaN,NaN,93.931911,2432.033634,-0.739491,NaN,NaN,NaN,NaN,NaN,NaN,0.143336,0.003723,NaN,0.453835,0.084885,NaN,0.231943,1.411599,87.108186,2499.539326,0.339021,80.495475,2662.352941,0.965656,0.082278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,pitcher,445276,2025,925,0.550270,0.427027,0.123243,0.432432,0.117838,0.563243,96.9,92.750964,83.039535,9.711429,2.6,153.50,0.3100,6.880131,0.2,1.405131,0.190,-0.123323,0.655352,2.673284,0.961436,0.815739,0.184261,0.627530,0.390560,0.337013,0.053547,0.303030,0.163636,0.418182,0.115152,-0.011016,NaN,NaN,NaN,92.710858,2592.426273,0.197212,NaN,NaN,NaN,93.107143,2320.154762,-0.659643,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.091703,0.814410,NaN,NaN,NaN,0.060044,0.669764,83.696364,2456.709091,0.602727,81.874194,2509.290323,1.034516,0.033843,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,pitcher,450203,2025,2605,0.467179,0.322457,0.144722,0.351248,0.115931,0.479463,97.3,93.068385,82.690887,10.377498,12.2,879.50,2.3600,6.155416,0.3,0.065130,1.560,0.082078,0.896490,2.236497,0.940283,0.865492,0.134508,0.596096,0.377818,0.379542,-0.001724,0.450704,0.251174,0.246479,0.051643,0.010534,87.516206,1815.205534,-1.140514,87.996522,2513.904348,0.072565,81.440061,3163.086066,1.280010,94.013506,2219.574026,-1.480130,94.208523,2322.633523,-0.995270,NaN,NaN,NaN,NaN,NaN,NaN,0.383046,0.151099,0.090267,0.276295,0.099294,NaN,NaN,1.454942,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [20]:
list(df.columns)

['role',
 'player_id',
 'year',
 'n_pitches_total',
 'batter_swing_rate',
 'batter_zone_swing_rate',
 'batter_chase_rate',
 'batter_contact_rate',
 'batter_whiff_rate',
 'in_zone_rate',
 'release_speed_max',
 'fastball_velo_mean',
 'offspeed_velo_mean',
 'velo_differential',
 'release_speed_iqr',
 'release_spin_rate_iqr',
 'pfx_x_iqr',
 'release_extension_mean',
 'release_extension_iqr',
 'pfx_z_mean',
 'pfx_z_iqr',
 'plate_x_mean',
 'plate_x_sd',
 'plate_z_mean',
 'plate_z_sd',
 'edge_percent',
 'meatball_percent',
 'first_pitch_strike_rate',
 'xwoba_allowed_lhb_mean',
 'xwoba_allowed_rhb_mean',
 'platoon_xwoba_allowed_diff',
 'gb_percent_allowed',
 'ld_percent_allowed',
 'fb_percent_allowed',
 'iffb_percent_allowed',
 'delta_run_exp_mean',
 'pt_CH_release_speed_mean',
 'pt_CH_release_spin_rate_mean',
 'pt_CH_pfx_x_mean',
 'pt_FC_release_speed_mean',
 'pt_FC_release_spin_rate_mean',
 'pt_FC_pfx_x_mean',
 'pt_CU_release_speed_mean',
 'pt_CU_release_spin_rate_mean',
 'pt_CU_pfx_x_mean',

### Batters

In [27]:
BUCKET = "diamond-dna"
FEATURE_PREFIX = "silver"

role = "batter"
year = "2025"

fs = s3fs.S3FileSystem()
path = f"s3://{BUCKET}/{FEATURE_PREFIX}/{role}/year={year}/player_year_features.parquet"
with fs.open(path, "rb") as f:
    df = pd.read_parquet(f)

pd.set_option("display.max_columns", None)
df.head()

,role,player_id,year,n_pitches_total,swing_rate,zone_swing_rate,chase_rate,contact_rate,whiff_rate,barrel_rate,hard_hit_rate,pull_percent,opposite_field_percent,gb_percent,ld_percent,fb_percent,iffb_percent,sweet_spot_percent,launch_speed_mean,launch_speed_iqr,launch_angle_mean,launch_angle_iqr,iso_value_mean,estimated_slg_using_speedangle_mean,woba_value_mean,estimated_woba_using_speedangle_mean,sprint_speed_mean,def_oaa_total,def_actual_fielding_success_rate_mean,def_adj_estimated_fielding_success_rate_mean,def_outfield_catch_completion_rate,def_arm_strength_max_mph,def_pop_time_2b_sec,def_framing_runs,def_drs_total
0,batter,456781,2025,774,0.512920,0.326873,0.186047,0.391473,0.121447,0.010830,0.223827,0.379310,0.282759,0.482759,0.248276,0.200000,0.068966,0.306859,83.162816,19.400,17.787004,42.00,0.095000,0.434493,0.301250,0.258718,25.0,1.0,0.79,0.77,NaN,72.1,NaN,NaN,NaN
1,batter,457705,2025,2344,0.453925,0.349829,0.104096,0.342577,0.111348,0.016598,0.257261,0.419355,0.182796,0.428954,0.219839,0.270777,0.080429,0.295172,83.360719,21.850,16.840000,45.00,0.106830,0.537488,0.307005,0.333453,27.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,batter,457759,2025,822,0.479319,0.336983,0.142336,0.403893,0.075426,0.006623,0.168874,0.321678,0.258741,0.384615,0.244755,0.286713,0.083916,0.294702,82.212252,18.075,24.238411,38.75,0.080808,0.467290,0.279040,0.298232,25.3,NaN,NaN,NaN,NaN,75.7,NaN,NaN,NaN
3,batter,467793,2025,2078,0.445621,0.321944,0.123677,0.345043,0.100577,0.007669,0.263804,0.502907,0.151163,0.409884,0.220930,0.261628,0.107558,0.279141,83.118558,23.200,18.113497,52.25,0.095808,0.447878,0.289421,0.290137,25.6,8.0,0.78,0.75,NaN,87.1,NaN,NaN,NaN
4,batter,500743,2025,1267,0.451460,0.315706,0.135754,0.376480,0.074980,0.015766,0.213964,0.368030,0.226766,0.457249,0.241636,0.234201,0.066914,0.295964,81.318694,20.650,14.941704,45.00,0.122093,0.434925,0.315407,0.296853,25.9,7.0,0.81,0.78,NaN,86.8,NaN,NaN,NaN


In [22]:
list(df.columns)

['role',
 'player_id',
 'year',
 'n_pitches_total',
 'swing_rate',
 'zone_swing_rate',
 'chase_rate',
 'contact_rate',
 'whiff_rate',
 'barrel_rate',
 'hard_hit_rate',
 'pull_percent',
 'opposite_field_percent',
 'gb_percent',
 'ld_percent',
 'fb_percent',
 'iffb_percent',
 'sweet_spot_percent',
 'launch_speed_mean',
 'launch_speed_iqr',
 'launch_angle_mean',
 'launch_angle_iqr',
 'iso_value_mean',
 'estimated_slg_using_speedangle_mean',
 'woba_value_mean',
 'estimated_woba_using_speedangle_mean',
 'sprint_speed_mean',
 'def_oaa_total',
 'def_actual_fielding_success_rate_mean',
 'def_adj_estimated_fielding_success_rate_mean',
 'def_outfield_catch_completion_rate',
 'def_arm_strength_max_mph',
 'def_pop_time_2b_sec',
 'def_framing_runs',
 'def_drs_total']

## Loading gold (preprocessed) player-year feature tables

Gold objects are written by `src.gold.silver_to_gold_preprocessing` using the same `{role}/year=Y/` partitions as silver, but the Parquet filename is **`player_year_features_preprocessed.parquet`** (see `gold_player_year_output_key` in `src/pipeline/lake_paths.py`). Each partition may also include **`preprocessing_metadata.json`** (column drops, NA handling, etc.).

Below we load the same role/year as the silver batter example for comparison.

In [7]:
BUCKET = "diamond-dna"
GOLD_PREFIX = "gold/statcast"  # default GOLD_PREFIX in src/pipeline/settings.py

role = "batter"
year = "2025"

fs = s3fs.S3FileSystem()
path = f"s3://{BUCKET}/{GOLD_PREFIX}/{role}/year={year}/player_year_features_preprocessed.parquet"
with fs.open(path, "rb") as f:
    df_gold = pd.read_parquet(f)

pd.set_option("display.max_columns", None)
df_gold.head()

TypeError: ClientArgsCreator.compute_endpoint_resolver_builtin_defaults() missing 1 required positional argument: 's3_disable_express_session_auth'

In [ ]:
import json

meta_path = f"s3://{BUCKET}/{GOLD_PREFIX}/{role}/year={year}/preprocessing_metadata.json"
try:
    with fs.open(meta_path, "r") as f:
        meta = json.load(f)
    print("Top-level keys:", sorted(meta.keys()))
    print("Row count:", meta.get("row_count"))
    print("Feature columns:", len(meta.get("feature_columns", [])))
except FileNotFoundError:
    print(f"No metadata at {meta_path} (run silver→gold preprocessing for this role/year first).")